In [11]:
import sqlite3
import pandas as pd

# Reconectar ao banco
conn = sqlite3.connect("lh_nautical.db")
print("Banco conectado!")

# Verificar se as tabelas estão lá
query = "SELECT name FROM sqlite_master WHERE type='table'"
print(pd.read_sql(query, conn))

Banco conectado!
            name
0         vendas
1       produtos
2       clientes
3         custos
4  base_completa


In [12]:
!pip install scikit-learn

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
query = """
SELECT
    id_client,
    id_product,
    SUM(qtd) AS total_comprado

FROM base_completa
GROUP BY id_client, id_product
"""

df_matriz = pd.read_sql(query, conn)

# Criar matriz cliente x produto
matriz = df_matriz.pivot_table(
    index="id_client",
    columns="id_product",
    values="total_comprado",
    fill_value=0
)

print("Matriz criada")
print(f"Clientes: {matriz.shape[0]}")
print(f"Produtos: {matriz.shape[1]}")

Matriz criada
Clientes: 49
Produtos: 150


In [14]:
# Calcular similaridade de cosseno entre clientes
similaridade = cosine_similarity(matriz)
df_similaridade = pd.DataFrame(
    similaridade,
    index=matriz.index,
    columns=matriz.index
)

print(df_similaridade.head())

id_client        1         2         3         4         5         6   \
id_client                                                               
1          1.000000  0.515989  0.521310  0.476556  0.464354  0.496652   
2          0.515989  1.000000  0.572834  0.534081  0.589122  0.546931   
3          0.521310  0.572834  1.000000  0.531108  0.587691  0.526124   
4          0.476556  0.534081  0.531108  1.000000  0.557748  0.593572   
5          0.464354  0.589122  0.587691  0.557748  1.000000  0.499733   

id_client        7         8         9         10  ...        40        41  \
id_client                                          ...                       
1          0.502182  0.608313  0.433643  0.543992  ...  0.440338  0.562179   
2          0.511147  0.477452  0.524593  0.485845  ...  0.566961  0.528855   
3          0.560818  0.538440  0.539319  0.554061  ...  0.551672  0.528177   
4          0.467565  0.562686  0.575782  0.582495  ...  0.570161  0.548155   
5          0.541748 

In [15]:
def recomendar_produtos(id_cliente, n_recomendacoes=5):
    # Verificar se cliente existe
    if id_cliente not in df_similaridade.index:
        return f"Cliente {id_cliente} não encontrado!"

    # Encontrar clientes mais similares
    similares = df_similaridade[id_cliente].sort_values(ascending=False)
    similares = similares.drop(id_cliente)  # remover o próprio cliente
    top_similares = similares.head(5).index.tolist()

    # Produtos que o cliente já comprou
    ja_comprou = set(matriz.loc[id_cliente][matriz.loc[id_cliente] > 0].index)

    # Produtos comprados pelos clientes similares
    recomendados = {}
    for cliente_similar in top_similares:
        produtos_similar = matriz.loc[cliente_similar]
        for produto, qtd in produtos_similar.items():
            if qtd > 0 and produto not in ja_comprou:
                recomendados[produto] = recomendados.get(produto, 0) + qtd

    # Ordenar por relevância
    recomendados = sorted(recomendados.items(), key=lambda x: x[1], reverse=True)
    top_produtos = [p[0] for p in recomendados[:n_recomendacoes]]

    return top_produtos

In [16]:
# Pegar o primeiro cliente como teste
cliente_teste = matriz.index[0]

recomendacoes = recomendar_produtos(cliente_teste, n_recomendacoes=5)

# Buscar nomes dos produtos recomendados
query = f"""
SELECT code, name, actual_category
FROM produtos
WHERE code IN ({','.join([str(p) for p in recomendacoes])})
"""

df_recomendacoes = pd.read_sql(query, conn)

print(f"Cliente: {cliente_teste}")
print(f"\nProdutos recomendados:")
print(df_recomendacoes.to_string(index=False))

Cliente: 1

Produtos recomendados:
 code                                       name actual_category
    8                                Gps Ais Zen     Eletronicos
   49                         Sonda Furuno Swift     Eletronicos
   94           Motor De Popa Volvo Magnum 276Hp       Propulsao
   96    Motor De Popa Tohatsu Boost Swift 126Hp       Propulsao
  115 Cabo De Nylon Delta Force Magnum Leviathan       Ancoragem


In [17]:
resultados = []

for cliente in matriz.index:
    recomendacoes = recomendar_produtos(cliente, n_recomendacoes=3)
    for produto in recomendacoes:
        resultados.append({
            "id_client": cliente,
            "id_product_recomendado": produto
        })

df_resultados = pd.DataFrame(resultados)

# Buscar dados dos produtos diretamente do banco
df_produtos_sql = pd.read_sql("SELECT code, name, price, actual_category FROM produtos", conn)

# Enriquecer com dados do banco
df_resultados = df_resultados.merge(
    df_produtos_sql[["code", "name", "price", "actual_category"]],
    left_on="id_product_recomendado",
    right_on="code",
    how="left"
).drop(columns=["code"])

print("Recomendações geradas!")
print(f"Total de recomendações: {len(df_resultados)}")
df_resultados.head(10)

Recomendações geradas!
Total de recomendações: 147


,id_client,id_product_recomendado,name,price,actual_category
0,1,49,Sonda Furuno Swift,13264.25,Eletronicos
1,1,94,Motor De Popa Volvo Magnum 276Hp,40750.84,Propulsao
2,1,96,Motor De Popa Tohatsu Boost Swift 126Hp,70620.84,Propulsao
3,2,67,Motor De Popa Yamaha Mako 108Hp,85789.05,Propulsao
4,2,73,Motor De Popa Torqeedo Core Hydra Flux 162Hp,143159.93,Propulsao
5,2,120,Boia De Arqueamento Danforth Flux Tidal,3124.49,Ancoragem
6,3,76,Motor Diesel Honda Aero 205Hp,148198.23,Propulsao
7,3,141,Boia De Arqueamento Bruce Nexus Abyss,4781.80,Ancoragem
8,3,19,Sonda Lowrance Tidal Storm Vox,16432.35,Eletronicos
9,4,6,Transponder Ais Vector,11820.21,Eletronicos


In [18]:
df_resultados.to_csv("recomendacoes.csv", index=False)

In [19]:
conn.close()